### Copyright (C) Infineon Technologies AG 2025
 
Copyright (c) 2025, Infineon Technologies AG, or an affiliate of Infineon Technologies AG. All rights reserved.
This software, associated documentation and materials ("Software") is owned by Infineon Technologies AG or one of its affiliates ("Infineon") and is protected by and subject to worldwide patent protection, worldwide copyright laws, and international treaty provisions. Therefore, you may use this Software only as provided in the license agreement accompanying the software package from which you obtained this Software. If no license agreement applies, then any use, reproduction, modification, translation, or compilation of this Software is prohibited without the express written permission of Infineon.

Disclaimer: UNLESS OTHERWISE EXPRESSLY AGREED WITH INFINEON, THIS SOFTWARE IS PROVIDED AS-IS, WITH NO WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING, BUT NOT LIMITED TO, ALL WARRANTIES OF NON-INFRINGEMENT OF THIRD-PARTY RIGHTS AND IMPLIED WARRANTIES SUCH AS WARRANTIES OF FITNESS FOR A SPECIFIC USE/PURPOSE OR MERCHANTABILITY. Infineon reserves the right to make changes to the Software without notice. You are responsible for properly designing, programming, and testing the functionality and safety of your intended application of the Software, as well as complying with any legal requirements related to its use. Infineon does not guarantee that the Software will be free from intrusion, data theft or loss, or other breaches ("Security Breaches"), and Infineon shall have no liability arising out of any Security Breaches. Unless otherwise explicitly approved by Infineon, the Software may not be used in any application where a failure of the Product or any consequences of the use thereof can reasonably be expected to result in personal injury.

# Vehicle Tracker

This example demonstrates an online multi-target vehicle-tracking pipeline using CARLA™-generated scenes and the RNN-based tracking approach from Milan et al. [1]. It trains two models: an RNN motion model that predicts vehicle states and an LSTM association model that assigns detections to tracks. The models are adapted for CARLA vehicle tracking and prepared for AURIX™ deployment.

## Notebook Structure

1. **Dataset overview and CARLA/MOT data preparation**
2. **Imports and helper functions**
3. **Visualize sample tracking data**
4. **Stage 1: RNN motion model training**
5. **Stage 2: LSTM association model training**
6. **Export and save both models**
7. **Convert the models for embedded deployment**
8. **Profile inference timing: RNN, then LSTM**
9. **Cross-platform benchmark comparison: RNN, then LSTM**
10. **References and Generated Artifacts**

## Dataset overview and CARLA/MOT data preparation

The tracking data are generated locally with CARLA Simulator version 0.9.16 and stored in the MOTChallenge format under `data/carla/`. CARLA is the scene generator; MOTChallenge provides the annotation format and tracking conventions. The dataset is split by town:

- **Train:** Town01, Town02, Town03, Town04, Town06, Town10HD
- **Validation:** Town05
- **Test:** Town07

Each `gt.txt` file contains frame-level bounding-box annotations:

`frame_id, track_id, x, y, w, h, 1, -1, -1, -1`

The fields identify the frame, vehicle track, bounding-box position and size, confidence, and reserved MOTChallenge fields. The data are generated for this example rather than downloaded as a fixed image dataset.

## Imports and helper functions

Load the shared tracking, data-loading, training, export, and conversion helpers.

In [ ]:
import sys
import os

parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from _CentralScripts.python_flask_client import CallTools
import _CentralScripts.helper_functions as cs
from modelling_helper import *

%matplotlib inline

device = cs.get_device()
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 3. Visualize sample tracking data

Visualize a sample frame from the training data (Town02) with ground-truth bounding boxes showing annotated vehicles. This checks the local CARLA/MOT annotations before model training.

In [ ]:
plot_sample_frame()

## Stage 1: RNN motion model training

Load CARLA `gt.txt` trajectories for the training and validation splits. Bounding boxes are normalized to `[-0.5, 0.5]` following Milan et al. [1]. The RNN predicts the next vehicle state from the current state and hidden state.

In [ ]:
train_ds = RealTrajectoryDataset(split="train", seq_length=SEQ_LENGTH)
val_ds = RealTrajectoryDataset(split="val", seq_length=SEQ_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train sequences: {:,}".format(len(train_ds)))
print("Val sequences  : {:,}".format(len(val_ds)))

Training the RNN motion model following Milan et al. [1]. The model predicts the next vehicle state from the current state and hidden state using Root Mean Square Propagation (RMSprop) optimizer with learning rate decay. Best model checkpoint is saved to `exported_model/checkpoints/`.

Training progress is printed every `val_freq` iterations:
- **Iter**: current training iteration
- **val loss**: Mean Squared Error (MSE) loss on validation set (lower is better)
- **best val loss**: lowest val loss achieved so far (saved as checkpoint)

In [ ]:
rnn_model, rnn_losses, rnn_vals = train_rnn(
    train_loader,
    val_loader,
    device,
    CHECKPOINT_DIR,
    max_iters=10000,
    val_freq=1000,
)

The graph shows the training loss (blue) and validation loss (orange) of the RNN motion model over iterations. The training loss starts from iteration 0 while the validation loss starts from iteration `val_freq` since validation is computed periodically.

In [ ]:
plot_training_curves(rnn_losses, rnn_vals, CHECKPOINT_DIR, mode="rnn")

Visualising the RNN motion model predictions against the ground truth trajectories. The model predicts the next bounding box position (x, y) from the current state. Red dashed lines show predictions, green solid lines show ground truth.

In [ ]:
plot_rnn_predictions(rnn_model, device, n_tracks=3)

## Stage 2: LSTM association model training

Load CARLA frames containing multiple tracks for association training. The LSTM learns assignment probabilities between predicted tracks and detections using the adapted association representation described by Milan et al. [1].

In [ ]:
lstm_train_ds = LSTMAssociationDataset(split="train", n_samples=100000)
lstm_val_ds = LSTMAssociationDataset(split="val", n_samples=10000)

lstm_train_loader = DataLoader(
    lstm_train_ds, batch_size=LSTM_BATCH_SIZE, shuffle=True, num_workers=0
)
lstm_val_loader = DataLoader(
    lstm_val_ds, batch_size=LSTM_BATCH_SIZE, shuffle=False, num_workers=0
)

print("LSTM Train samples: {:,}".format(len(lstm_train_ds)))
print("LSTM Val samples  : {:,}".format(len(lstm_val_ds)))

Training the LSTM association model following Milan et al. [1] with hidden size of 64. Three domain adaptations [4] are applied for CARLA vehicle tracking:
- **Row-based input**: one row of C matrix per LSTM step instead of full matrix
- **Distance metric**: normalized distance [0,1] instead of raw Euclidean distances
- **Loss normalization**: renormalized over real detections only to avoid padding dilution

Training uses Root Mean Square Propagation (RMSprop) optimizer with learning rate decay. Best model checkpoint is saved to `exported_model/checkpoints/`.

Training progress is printed every `val_freq` iterations:
- **Iter**: current training iteration
- **val loss**: Negative Log Likelihood (NLL) loss on validation set (lower is better)
- **val accuracy**: percentage of correct vehicle-detection assignments (higher is better)
- **best val loss**: lowest val loss achieved so far (saved as checkpoint)

**Warning: the training may take longer than 1 h.**

In [ ]:
lstm_model, lstm_losses, lstm_vals = train_lstm(
    lstm_train_loader,
    lstm_val_loader,
    device,
    CHECKPOINT_DIR,
    max_iters=2000,
    val_freq=200,
)

The graph shows the training loss (blue) and validation loss (orange) of the LSTM association model over iterations. The training loss starts from iteration 0 while the validation loss starts from iteration `val_freq` since validation is computed periodically. The decreasing loss indicates the model is learning to correctly assign detections to tracks.

In [ ]:
plot_training_curves(lstm_losses, lstm_vals, CHECKPOINT_DIR, mode="lstm")

Visualising the LSTM association model output. Each cell shows the assignment probability between a track (row) and a detection (column). Darker cells indicate higher probability of assignment. Red boxes show the ground truth correct assignments and blue dashed boxes show the LSTM predictions. When red and blue boxes overlap the model correctly assigned the detection to its track.

In [ ]:
test_and_plot_lstm(lstm_model, lstm_val_loader, device)

## Export and save both models

Export the RNN motion model and LSTM association model to ONNX using representative CARLA data as input. Store each model with its input and reference output artifacts in `out/`.

In [ ]:
model_names = export_models(
    rnn_model,
    lstm_model,
    val_loader,
    lstm_val_loader,
    cs,
)

## 7. Convert the models for embedded deployment

Compile both exported models for AURIX™ and TRAVEO™ deployment targets. The supported targets are AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4. Submit each model with representative input/output data and retain generated code, binaries, logs, and timing results.

In [ ]:
cs.ensure_docker_container()

In [ ]:
target = "tc4dx_ppu"  # Available targets: "tc3xx", "tc4dx", "tc4dx_ppu", "arm_m4"
for model_name in model_names:
    model_folder, onnx_model_file = cs.get_output_paths(model_name)
    tool = CallTools(
        folder=model_folder,
        url="http://localhost:8080/convert",
        target=target,
        profile=True,
    )
    tool.convert_model()

## Profiling and inference timing

The first benchmark tier profiles each converted model on the selected target. The order is intentional: the RNN motion model is displayed first, followed by the LSTM association model. Each result includes estimated cycles, instructions, average CPI, latency, throughput, and per-node profile shares.

### RNN Model: profiling

In [ ]:
print(f"Displaying inference benchmark for model: {model_names[0]}")
model_folder, onnx_model_file = cs.get_output_paths(model_names[0])
cs.display_inference_benchmark(model_folder, target)

In [ ]:
cs.plot_profile_shares(model_folder, target)

### LSTM Model: profiling

In [ ]:
print(f"Displaying inference benchmark for model: {model_names[1]}")
model_folder, onnx_model_file = cs.get_output_paths(model_names[1])
cs.display_inference_benchmark(model_folder, target)

In [ ]:
cs.plot_profile_shares(model_folder, target)

## Cross-platform benchmark comparison

The second benchmark tier converts and compares the models across all supported hardware platforms. The order is intentional: the RNN motion model is displayed first, followed by the LSTM association model.

### RNN model: cross-platform comparison

In [ ]:
print(f"Displaying inference benchmark for model: {model_names[0]}")
model_folder, onnx_model_file = cs.get_output_paths(model_names[0])

target = "compare"

tool = CallTools(
    folder=model_folder,
    url="http://localhost:8080/convert",
    target=target,
)
tool.convert_model()

In [ ]:
df = cs.display_platform_comparison(model_folder)

### LSTM model: cross-platform comparison

In [ ]:
print(f"Displaying inference benchmark for model: {model_names[1]}")
model_folder, onnx_model_file = cs.get_output_paths(model_names[1])

target = "compare"

tool = CallTools(
    folder=model_folder,
    url="http://localhost:8080/convert",
    target=target,
)
tool.convert_model()

In [ ]:
df = cs.display_platform_comparison(model_folder)

## 10. References and Generated Artifacts

- **Method:** Milan et al., “Online Multi-Target Tracking Using Recurrent Neural Networks.” [1]
- **Scene generator:** CARLA Simulator [2], version 0.9.16, used to generate the local vehicle-tracking scenes.
- **Annotation convention:** MOTChallenge format and tracking conventions [3].
- **Model adaptation:** Shamsuddin, “Vision-Based Vehicle Tracking Algorithm for Automated Driving” [4].
- **Artifacts:** RNN and LSTM ONNX models, representative CARLA inputs and outputs, generated code, binaries, logs, profiling results, and cross-platform benchmark results, saved under `out/vehicle_tracker_rnn/test_vehicle_tracker_rnn/<target>/` and `out/vehicle_tracker_lstm/test_vehicle_tracker_lstm/<target>/`.
- **Deployment targets:** AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4, with per-model RNN and LSTM timing and platform-comparison results displayed above.

[1]: Milan, A. et al. (2017). "Online Multi-Target Tracking Using Recurrent Neural Networks.", AAAI. https://arxiv.org/abs/1604.03635<br>
[2]: CARLA Simulator. https://carla.org<br>
[3]: MOTChallenge. https://motchallenge.net<br>
[4]: Shamsuddin, R. (2026). "Vision-Based Vehicle Tracking Algorithm for Automated Driving", master thesis, TUD Dresden University of Technology